# HESE Event Comparison

1. Load EvtGen events from v1 and v2 processing.
2. Compare v1 vs v2 at event level.
3. Load Emre's L5 run list.
4. Three-way run-level comparison: v1, v2, L5.

In [1]:
import re
import tables
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load EvtGen events — v1 and v2

In [2]:
BASE_PATH = "/data/user/tvaneede/GlobalFit/reco_processing/data/hese/output/{version}"

DATASETS = [
    "IC79_2010",
    "IC86_2011",
    "IC86_2012",
    "IC86_2013",
    "IC86_2014",
    "IC86_2015",
    "IC86_2016",
    "IC86_2017",
    "IC86_2018",
    "IC86_2019",
    "IC86_2020",
    "IC86_2021",
    "IC86_2022",
]


def load_evtgen(version):
    records = []
    for dataset in DATASETS:
        path = Path(BASE_PATH.format(version=version)) / dataset / "EvtGen" / "EvtGen.h5"
        if not path.exists():
            print(f"  {dataset}: MISSING")
            continue
        with tables.open_file(str(path), "r") as f:
            df_hdr = pd.DataFrame({
                "run":   f.root.I3EventHeader.col("Run"),
                "event": f.root.I3EventHeader.col("Event"),
            })
            df_en = pd.DataFrame({
                "run":         f.root.RecoETot.col("Run"),
                "event":       f.root.RecoETot.col("Event"),
                "reco_energy": f.root.RecoETot.col("value"),
            })
            # HESE_CausalQTot is present in v2 and newer v1 datasets (IC86_2022+)
            if hasattr(f.root, "HESE_CausalQTot"):
                df_qtot = pd.DataFrame({
                    "run":              f.root.HESE_CausalQTot.col("Run"),
                    "event":            f.root.HESE_CausalQTot.col("Event"),
                    "hese_causal_qtot": f.root.HESE_CausalQTot.col("value"),
                })
            else:
                df_qtot = None
        df = df_hdr.merge(df_en, on=["run", "event"], how="left")
        if df_qtot is not None:
            df = df.merge(df_qtot, on=["run", "event"], how="left")
        df.insert(0, "dataset", dataset)
        records.append(df)
        print(f"  {dataset}: {len(df)} events")
    result = pd.concat(records, ignore_index=True)
    print(f"  Total: {len(result)}")
    return result

In [3]:
print("=== v1 ===")
df_v1 = load_evtgen("v1")

print("\n=== v2 ===")
df_v2 = load_evtgen("v2")

=== v1 ===
  IC79_2010: 7 events
  IC86_2011: 19 events
  IC86_2012: 8 events
  IC86_2013: 13 events
  IC86_2014: 15 events
  IC86_2015: 8 events
  IC86_2016: 22 events
  IC86_2017: 18 events
  IC86_2018: 16 events
  IC86_2019: 19 events
  IC86_2020: 9 events
  IC86_2021: 12 events
  IC86_2022: 19 events
  Total: 185

=== v2 ===
  IC79_2010: 7 events
  IC86_2011: 21 events
  IC86_2012: 9 events
  IC86_2013: 16 events
  IC86_2014: 15 events
  IC86_2015: 9 events
  IC86_2016: 22 events
  IC86_2017: 18 events
  IC86_2018: 16 events
  IC86_2019: 19 events
  IC86_2020: 10 events
  IC86_2021: 12 events
  IC86_2022: 19 events
  Total: 193


## 2. v1 vs v2 — event-level comparison

In [4]:
set_v1 = set(zip(df_v1["dataset"], df_v1["run"], df_v1["event"]))
set_v2 = set(zip(df_v2["dataset"], df_v2["run"], df_v2["event"]))

print(f"Events in v1:          {len(set_v1)}")
print(f"Events in v2:          {len(set_v2)}")
print(f"Events in both:        {len(set_v1 & set_v2)}")
print(f"Only in v1 (not v2):   {len(set_v1 - set_v2)}")
print(f"Only in v2 (not v1):   {len(set_v2 - set_v1)}")

Events in v1:          185
Events in v2:          193
Events in both:        184
Only in v1 (not v2):   1
Only in v2 (not v1):   9


In [5]:
# Per-dataset summary
all_datasets = sorted(set(df_v1["dataset"]) | set(df_v2["dataset"]))
rows = []
for ds in all_datasets:
    s1 = set(zip(df_v1.loc[df_v1["dataset"] == ds, "run"], df_v1.loc[df_v1["dataset"] == ds, "event"]))
    s2 = set(zip(df_v2.loc[df_v2["dataset"] == ds, "run"], df_v2.loc[df_v2["dataset"] == ds, "event"]))
    rows.append({
        "dataset":   ds,
        "n_v1":      len(s1),
        "n_v2":      len(s2),
        "in_both":   len(s1 & s2),
        "only_v1":   len(s1 - s2),
        "only_v2":   len(s2 - s1),
    })
pd.DataFrame(rows).set_index("dataset")

,n_v1,n_v2,in_both,only_v1,only_v2
dataset,,,,,
IC79_2010,7,7,7,0,0
IC86_2011,19,21,19,0,2
IC86_2012,8,9,8,0,1
IC86_2013,13,16,13,0,3
IC86_2014,15,15,14,1,1
IC86_2015,8,9,8,0,1
IC86_2016,22,22,22,0,0
IC86_2017,18,18,18,0,0
IC86_2018,16,16,16,0,0


In [7]:
# Events only in v1
df_v1["in_v2"] = [k in set_v2 for k in zip(df_v1["dataset"], df_v1["run"], df_v1["event"])]
only_v1 = df_v1[~df_v1["in_v2"]][["dataset", "run", "event", "reco_energy"]].reset_index(drop=True)
print(f"Events only in v1: {len(only_v1)}")
only_v1

Events only in v1: 1


,dataset,run,event,reco_energy
0,IC86_2014,125914,75630389,72763.011514


In [8]:
# Events only in v2 — with HESE_CausalQTot
df_v2["in_v1"] = [k in set_v1 for k in zip(df_v2["dataset"], df_v2["run"], df_v2["event"])]

cols = ["dataset", "run", "event", "reco_energy"]
if "hese_causal_qtot" in df_v2.columns:
    cols.append("hese_causal_qtot")

only_v2 = df_v2[~df_v2["in_v1"]][cols].reset_index(drop=True)
print(f"Events only in v2: {len(only_v2)}")
display(only_v2)

Events only in v2: 9


,dataset,run,event,reco_energy,hese_causal_qtot
0,IC86_2011,119311,430943,3.269002e+05,14156.095864
1,IC86_2011,119583,141609,5.709847e+04,6875.744233
2,IC86_2012,121947,7181486,7.449678e+04,9355.539456
3,IC86_2013,122752,41309299,1.482176e+05,19691.222671
4,IC86_2013,123770,442256,4.169044e+06,6157.379055
5,IC86_2013,122604,17469985,2.250298e+05,18581.193399
6,IC86_2014,126359,9400616,2.526825e+04,6008.448549
7,IC86_2015,127751,927145,2.326454e+05,8745.913758
8,IC86_2020,134994,1103075,3.565653e+05,10871.700004


In [9]:
# Check if the v2-only runs are present in Neha's folders
NEHA_ALL   = Path("/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/HESE")
NEHA_EVENT = Path("/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/i3files/NoDeepCore/HESE12/Bfr")

def in_neha_all(dataset, run):
    # files named Run{run:08d}.i3.zst
    return (NEHA_ALL / dataset / f"Run{run:08d}.i3.zst").exists()

def in_neha_event(dataset, run):
    # files named Run{run:08d}_*
    folder = NEHA_EVENT / dataset
    return folder.exists() and any(folder.glob(f"Run{run:08d}_*"))

only_v2["in_neha_all"]   = [in_neha_all(ds, r)   for ds, r in zip(only_v2["dataset"], only_v2["run"])]
only_v2["in_neha_event"] = [in_neha_event(ds, r) for ds, r in zip(only_v2["dataset"], only_v2["run"])]

display(only_v2)

,dataset,run,event,reco_energy,hese_causal_qtot,in_neha_all,in_neha_event
0,IC86_2011,119311,430943,3.269002e+05,14156.095864,False,False
1,IC86_2011,119583,141609,5.709847e+04,6875.744233,False,False
2,IC86_2012,121947,7181486,7.449678e+04,9355.539456,True,False
3,IC86_2013,122752,41309299,1.482176e+05,19691.222671,False,True
4,IC86_2013,123770,442256,4.169044e+06,6157.379055,False,False
5,IC86_2013,122604,17469985,2.250298e+05,18581.193399,False,True
6,IC86_2014,126359,9400616,2.526825e+04,6008.448549,True,False
7,IC86_2015,127751,927145,2.326454e+05,8745.913758,False,False
8,IC86_2020,134994,1103075,3.565653e+05,10871.700004,False,False


## 3. Load Emre's L5 run list

In [10]:
L5_BASE  = "/data/user/eyildizci/hese_tracks_processing/L5"
L5_YEARS = [str(y) for y in range(2011, 2023)]

l5_records = []
for year in L5_YEARS:
    folder = Path(L5_BASE) / f"IC86_{year}"
    for fname in sorted(folder.iterdir()):
        m = re.search(r"Run(\d+)", fname.name)
        if m:
            l5_records.append({"dataset": f"IC86_{year}", "run": int(m.group(1))})

df_l5 = pd.DataFrame(l5_records)
print(f"L5 total files (runs): {len(df_l5)}")
df_l5.groupby("dataset").size().rename("n_files")

L5 total files (runs): 179


dataset
IC86_2011    19
IC86_2012     8
IC86_2013    15
IC86_2014    15
IC86_2015     8
IC86_2016    22
IC86_2017    18
IC86_2018    16
IC86_2019    18
IC86_2020     9
IC86_2021    12
IC86_2022    19
Name: n_files, dtype: int64

## 4. Three-way run-level comparison: v1 · v2 · L5

In [11]:
# Unique (dataset, run) sets for each source
runs_v1  = set(zip(df_v1["dataset"],  df_v1["run"]))
runs_v2  = set(zip(df_v2["dataset"],  df_v2["run"]))
runs_l5  = set(zip(df_l5["dataset"],  df_l5["run"]))

all_runs = runs_v1 | runs_v2 | runs_l5

# Build a master table: one row per unique (dataset, run)
df_runs = pd.DataFrame(sorted(all_runs), columns=["dataset", "run"])
df_runs["in_v1"] = [k in runs_v1 for k in zip(df_runs["dataset"], df_runs["run"])]
df_runs["in_v2"] = [k in runs_v2 for k in zip(df_runs["dataset"], df_runs["run"])]
df_runs["in_l5"] = [k in runs_l5 for k in zip(df_runs["dataset"], df_runs["run"])]

print(f"Total unique (dataset, run) pairs: {len(df_runs)}")
df_runs.head(10)

Total unique (dataset, run) pairs: 193


,dataset,run,in_v1,in_v2,in_l5
0,IC79_2010,115994,True,True,False
1,IC79_2010,116528,True,True,False
2,IC79_2010,116698,True,True,False
3,IC79_2010,117371,True,True,False
4,IC79_2010,117782,True,True,False
5,IC79_2010,118145,True,True,False
6,IC86_2011,118178,True,True,True
7,IC86_2011,118283,True,True,True
8,IC86_2011,118381,True,True,True
9,IC86_2011,118435,True,True,True


In [12]:
# Summary: 7-subset breakdown
subsets = [
    (True,  True,  True,  "v1 ∩ v2 ∩ L5  (all three)"),
    (True,  True,  False, "v1 ∩ v2 only   (not L5)"),
    (True,  False, True,  "v1 ∩ L5 only   (not v2)"),
    (False, True,  True,  "v2 ∩ L5 only   (not v1)"),
    (True,  False, False, "v1 only"),
    (False, True,  False, "v2 only"),
    (False, False, True,  "L5 only"),
]

for v1, v2, l5, label in subsets:
    mask = (df_runs["in_v1"] == v1) & (df_runs["in_v2"] == v2) & (df_runs["in_l5"] == l5)
    print(f"{mask.sum():3d}  {label}")

175  v1 ∩ v2 ∩ L5  (all three)
  7  v1 ∩ v2 only   (not L5)
  1  v1 ∩ L5 only   (not v2)
  2  v2 ∩ L5 only   (not v1)
  0  v1 only
  7  v2 only
  1  L5 only


In [13]:
# Per-dataset breakdown
rows = []
for ds in sorted(all_runs, key=lambda x: x[0]):
    pass  # just need unique datasets

all_ds = sorted(df_runs["dataset"].unique())
rows = []
for ds in all_ds:
    sub = df_runs[df_runs["dataset"] == ds]
    rows.append({
        "dataset":        ds,
        "v1∩v2∩L5":       int((sub["in_v1"] & sub["in_v2"] & sub["in_l5"]).sum()),
        "v1∩v2 only":     int((sub["in_v1"] & sub["in_v2"] & ~sub["in_l5"]).sum()),
        "v1∩L5 only":     int((sub["in_v1"] & ~sub["in_v2"] & sub["in_l5"]).sum()),
        "v2∩L5 only":     int((~sub["in_v1"] & sub["in_v2"] & sub["in_l5"]).sum()),
        "v1 only":        int((sub["in_v1"] & ~sub["in_v2"] & ~sub["in_l5"]).sum()),
        "v2 only":        int((~sub["in_v1"] & sub["in_v2"] & ~sub["in_l5"]).sum()),
        "L5 only":        int((~sub["in_v1"] & ~sub["in_v2"] & sub["in_l5"]).sum()),
    })
pd.DataFrame(rows).set_index("dataset")

,v1∩v2∩L5,v1∩v2 only,v1∩L5 only,v2∩L5 only,v1 only,v2 only,L5 only
dataset,,,,,,,
IC79_2010,0,6,0,0,0,0,0
IC86_2011,19,0,0,0,0,2,0
IC86_2012,8,0,0,0,0,1,0
IC86_2013,13,0,0,2,0,1,0
IC86_2014,13,1,1,0,0,1,1
IC86_2015,8,0,0,0,0,1,0
IC86_2016,22,0,0,0,0,0,0
IC86_2017,18,0,0,0,0,0,0
IC86_2018,16,0,0,0,0,0,0


In [14]:
# Runs not in all three — full detail
incomplete = df_runs[~(df_runs["in_v1"] & df_runs["in_v2"] & df_runs["in_l5"])].reset_index(drop=True)
print(f"Runs not present in all three sources: {len(incomplete)}")
incomplete

Runs not present in all three sources: 18


,dataset,run,in_v1,in_v2,in_l5
0,IC79_2010,115994,True,True,False
1,IC79_2010,116528,True,True,False
2,IC79_2010,116698,True,True,False
3,IC79_2010,117371,True,True,False
4,IC79_2010,117782,True,True,False
5,IC79_2010,118145,True,True,False
6,IC86_2011,119311,False,True,False
7,IC86_2011,119583,False,True,False
8,IC86_2012,121947,False,True,False
9,IC86_2013,122604,False,True,True


---
## 5. v2 benchmark — apply reco_energy > 60 TeV cut

In [15]:
ENERGY_CUT = 60e3  # GeV

df_v2_cut = df_v2[df_v2["reco_energy"] > ENERGY_CUT].reset_index(drop=True)
print(f"v2 events before cut: {len(df_v2)}")
print(f"v2 events after cut:  {len(df_v2_cut)}")
df_v2_cut.groupby("dataset").size().rename("n_events")

v2 events before cut: 193
v2 events after cut:  115


dataset
IC79_2010     3
IC86_2011    12
IC86_2012     4
IC86_2013    12
IC86_2014    11
IC86_2015     7
IC86_2016    13
IC86_2017    12
IC86_2018     7
IC86_2019     8
IC86_2020     7
IC86_2021     7
IC86_2022    12
Name: n_events, dtype: int64

## 6. HESE12 hdf5 files — verify topology sub-files add up

In [16]:
BFR = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr"

HESE12_FILES = {
    "Cascades":       f"{BFR}/HESE12_Cascades.hdf5",
    "DoubleCascades": f"{BFR}/HESE12_DoubleCascades.hdf5",
    "Tracks":         f"{BFR}/HESE12_Tracks.hdf5",
    "Total":          f"{BFR}/HESE12.hdf5",
}


def load_hese12(path):
    with tables.open_file(path, "r") as f:
        df_hdr = pd.DataFrame({
            "run":   f.root.I3EventHeader.col("Run"),
            "event": f.root.I3EventHeader.col("Event"),
        })
        df_en = pd.DataFrame({
            "run":         f.root.RecoETot.col("Run"),
            "event":       f.root.RecoETot.col("Event"),
            "reco_energy": f.root.RecoETot.col("value"),
        })
    return df_hdr.merge(df_en, on=["run", "event"], how="left")


hese12 = {name: load_hese12(path) for name, path in HESE12_FILES.items()}

for name, df in hese12.items():
    print(f"{name:15s}: {len(df)} events")

# Check topology files sum to total
n_topo  = len(hese12["Cascades"]) + len(hese12["DoubleCascades"]) + len(hese12["Tracks"])
n_total = len(hese12["Total"])
print(f"\nCascades + DoubleCascades + Tracks = {n_topo}")
print(f"HESE12.hdf5 total                  = {n_total}")
print(f"Match: {n_topo == n_total}")

# Verify at event level
set_topo  = (set(zip(hese12["Cascades"]["run"],       hese12["Cascades"]["event"]))
           | set(zip(hese12["DoubleCascades"]["run"],  hese12["DoubleCascades"]["event"]))
           | set(zip(hese12["Tracks"]["run"],          hese12["Tracks"]["event"])))
set_total = set(zip(hese12["Total"]["run"], hese12["Total"]["event"]))

only_topo  = set_topo  - set_total
only_total = set_total - set_topo
print(f"\nEvents in sub-files but not in HESE12.hdf5: {len(only_topo)}")
print(f"Events in HESE12.hdf5 but not in sub-files: {len(only_total)}")

Cascades       : 64 events
DoubleCascades : 5 events
Tracks         : 28 events
Total          : 97 events

Cascades + DoubleCascades + Tracks = 97
HESE12.hdf5 total                  = 97
Match: True

Events in sub-files but not in HESE12.hdf5: 0
Events in HESE12.hdf5 but not in sub-files: 0


## 7. Compare v2 (after cut) with HESE12.hdf5

In [17]:
df_hese12 = hese12["Total"]
set_hese12 = set(zip(df_hese12["run"], df_hese12["event"]))
set_v2_cut = set(zip(df_v2_cut["run"], df_v2_cut["event"]))

print(f"v2 events (after cut): {len(set_v2_cut)}")
print(f"HESE12.hdf5 events:    {len(set_hese12)}")
print(f"In both:               {len(set_v2_cut & set_hese12)}")
print(f"Only in v2 (not HESE12): {len(set_v2_cut - set_hese12)}")
print(f"Only in HESE12 (not v2): {len(set_hese12 - set_v2_cut)}")

v2 events (after cut): 115
HESE12.hdf5 events:    97
In both:               95
Only in v2 (not HESE12): 20
Only in HESE12 (not v2): 2


In [18]:
# Per-dataset summary
df_v2_cut["in_hese12"] = [k in set_hese12 for k in zip(df_v2_cut["run"], df_v2_cut["event"])]
summary = df_v2_cut.groupby("dataset")["in_hese12"].agg(n_events="count", n_in_hese12="sum")
summary["fraction"] = summary["n_in_hese12"] / summary["n_events"]
print(summary.to_string())

# v2 events (after cut) missing from HESE12
cols = ["dataset", "run", "event", "reco_energy"]
if "hese_causal_qtot" in df_v2_cut.columns:
    cols.append("hese_causal_qtot")

only_v2_cut = df_v2_cut[~df_v2_cut["in_hese12"]][cols].reset_index(drop=True)
print(f"\nv2 events (after cut) not in HESE12: {len(only_v2_cut)}")
display(only_v2_cut)

# HESE12 events missing from v2 (after cut)
df_hese12["in_v2_cut"] = [k in set_v2_cut for k in zip(df_hese12["run"], df_hese12["event"])]
only_hese12 = df_hese12[~df_hese12["in_v2_cut"]][["run", "event", "reco_energy"]].reset_index(drop=True)
print(f"\nHESE12 events not in v2 (after cut): {len(only_hese12)}")
display(only_hese12)

           n_events  n_in_hese12  fraction
dataset                                   
IC79_2010         3            3  1.000000
IC86_2011        12           11  0.916667
IC86_2012         4            3  0.750000
IC86_2013        12           10  0.833333
IC86_2014        11           10  0.909091
IC86_2015         7            6  0.857143
IC86_2016        13           12  0.923077
IC86_2017        12           12  1.000000
IC86_2018         7            7  1.000000
IC86_2019         8            8  1.000000
IC86_2020         7            6  0.857143
IC86_2021         7            7  1.000000
IC86_2022        12            0  0.000000

v2 events (after cut) not in HESE12: 20


,dataset,run,event,reco_energy,hese_causal_qtot
0,IC86_2011,119311,430943,3.269002e+05,14156.095864
1,IC86_2012,121947,7181486,7.449678e+04,9355.539456
2,IC86_2013,123770,442256,4.169044e+06,6157.379055
3,IC86_2013,122604,17469985,2.250298e+05,18581.193399
4,IC86_2014,125826,470241,4.164990e+05,38785.083770
5,IC86_2015,127751,927145,2.326454e+05,8745.913758
6,IC86_2016,128672,38561326,8.313672e+04,7517.631427
7,IC86_2020,134994,1103075,3.565653e+05,10871.700004
8,IC86_2022,137845,43811235,9.899828e+04,12903.049979
9,IC86_2022,138125,11333473,1.800607e+05,20403.775029



HESE12 events not in v2 (after cut): 2


,run,event,reco_energy
0,125914,75630389,70976.904502
1,127210,51027948,60806.979712
